# Scraper de Metrocuadrado — arriendos en Cali

Este notebook usa Playwright porque los anuncios se cargan dinámicamente con JavaScript. Extrae los enlaces y el texto visible de los anuncios, intenta cargar más resultados mediante desplazamiento y guarda los datos en CSV y Excel.

Usa el scraper con pausas razonables y respeta los términos de uso y las restricciones del sitio.

## 1. Instalar dependencias

Ejecuta esta celda una sola vez. Después reinicia el kernel si Jupyter lo solicita.

In [2]:
%pip install playwright pandas openpyxl
!python -m playwright install chromium

Note: you may need to restart the kernel to use updated packages.


## 2. Configuración

In [3]:
import re
import sys
import asyncio
import threading
import pandas as pd
from urllib.parse import urljoin
from playwright.async_api import async_playwright

URL = (
    "https://www.metrocuadrado.com/"
    "arriendo/?search=form"
)
BASE_URL = "https://www.metrocuadrado.com"
MAX_SCROLLS = 6000
PAUSA_ENTRE_SCROLLS_MS = 500
MAX_PAGINAS = 199  # None para recorrer todas las páginas del paginador


def ejecutar_coroutine_en_hilo(coro_factory):
    """Ejecuta una corrutina en un hilo aparte con su propio event loop.

    En Windows, el kernel de Jupyter usa SelectorEventLoop (por compatibilidad
    con Tornado), pero Playwright necesita ProactorEventLoop para poder lanzar
    el subproceso del navegador. Si se hace `await` directamente en la celda,
    falla con `NotImplementedError`. Este helper corre la corrutina en un hilo
    nuevo con ProactorEventLoop (en Windows) para evitar el conflicto.
    """
    resultado = {}
    error = {}

    def _run():
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            resultado["valor"] = loop.run_until_complete(coro_factory())
        except Exception as exc:
            error["excepcion"] = exc
        finally:
            loop.close()

    hilo = threading.Thread(target=_run)
    hilo.start()
    hilo.join()

    if "excepcion" in error:
        raise error["excepcion"]
    return resultado["valor"]

## 3. Funciones del scraper

El selector principal busca enlaces cuyo destino contiene `/inmueble/`. Es más resistente que depender de nombres de clases CSS que pueden cambiar.

In [4]:
def limpiar_texto(texto):
    return " ".join((texto or "").split())


def extraer_campos(texto, alt=""):
    """Extrae precio, área, habitaciones, baños, parqueaderos, tipo, sector y ciudad.

    El texto visible del enlace (inner_text) NO incluye habitaciones/baños/área/
    garajes: esos datos los renderiza un web component (<pt-main-specs>) sin
    texto accesible. Sí aparecen, en cambio, en el atributo `alt` de la foto de
    la tarjeta, con un formato consistente, p. ej.:
    "Foto de Casa en Venta en PANCE, Cali con 4 habitaciones, 5 baños,
    área 350 m2, 4 garaje - 3566-M5719054". Por eso se combinan ambas fuentes.

    El sector y la ciudad se extraen del título del anuncio, que sigue el
    patrón "<Tipo> en Venta, <Sector>, <Ciudad>" (p. ej. "Casa en Venta,
    PANCE AV LA MARIA, Cali").
    """
    texto = limpiar_texto(texto)
    alt = limpiar_texto(alt)
    combinado = f"{texto} {alt}".strip()

    def buscar(*patrones):
        for patron in patrones:
            coincidencia = re.search(patron, combinado, flags=re.IGNORECASE)
            if coincidencia:
                return coincidencia.group(1)
        return None

    ubicacion = re.search(r"en Arriendo,\s*([^,]+),\s*([^,\n]+)", combinado, flags=re.IGNORECASE)
    sector = ubicacion.group(1).strip() if ubicacion else None
    ciudad = ubicacion.group(2).strip() if ubicacion else None

    return {
        "precio_texto": buscar(r"(\$\s*[\d\.,]+)"),
        "area_m2": buscar(r"área\s*([\d\.,]+)\s*m[²2]", r"([\d\.,]+)\s*m[²2]"),
        "habitaciones": buscar(r"(\d+)\s*habitac"),
        "banos": buscar(r"(\d+)\s*ba[ñn]o"),
        "parqueaderos": buscar(r"(\d+)\s*garaje", r"(\d+)\s*par(?:queadero)?"),
        "tipo": buscar(r"((?:Apartamento|Casa|Apartaestudio|Finca)\s+en\s+Arriendo)"),
        "sector": sector,
        "ciudad": ciudad,
    }


async def recolectar_anuncios_de_pagina(page, anuncios, max_scrolls):
    """Recorre la página actual con scroll hasta que dejan de aparecer anuncios nuevos.

    Los anuncios ya vistos (por href) se saltan sin volver a pedirle al navegador
    su inner_text ni el alt de la imagen: esas dos llamadas son las que más
    tardan (van y vuelven al proceso del navegador), y repetirlas en cada
    scroll para anuncios que no cambian es lo que hacía lenta cada página.
    """
    sin_cambios = 0
    cantidad_anterior = len(anuncios)
    hrefs_vistos = set(anuncios.keys())

    for intento in range(max_scrolls):
        enlaces = page.locator('a[href*="/inmueble/"]')
        cantidad = await enlaces.count()

        for i in range(cantidad):
            enlace = enlaces.nth(i)
            href = await enlace.get_attribute("href")
            if not href:
                continue

            url_anuncio = urljoin(BASE_URL, href)
            if url_anuncio in hrefs_vistos:
                continue

            texto = await enlace.inner_text()
            try:
                alt = await enlace.locator("img").first.get_attribute("alt", timeout=1_000)
            except Exception:
                alt = ""

            texto = limpiar_texto(texto)
            anuncios[url_anuncio] = {
                "url": url_anuncio,
                "texto": texto,
                **extraer_campos(texto, alt),
            }
            hrefs_vistos.add(url_anuncio)

        print(f"  Intento {intento + 1}: {len(anuncios)} anuncios en total")
        await page.evaluate("window.scrollBy(0, 1200)")
        await page.wait_for_timeout(PAUSA_ENTRE_SCROLLS_MS)

        if len(anuncios) == cantidad_anterior:
            sin_cambios += 1
        else:
            sin_cambios = 0

        cantidad_anterior = len(anuncios)
        if sin_cambios >= 2:
            break


async def ir_a_siguiente_pagina(page):
    """Hace clic en el botón "siguiente" del paginador. Devuelve False si no hay más páginas."""
    boton_siguiente = page.locator(".rc-pagination-next")

    if await boton_siguiente.count() == 0:
        return False

    deshabilitado = await boton_siguiente.get_attribute("aria-disabled")
    if deshabilitado == "true":
        return False

    try:
        await boton_siguiente.locator("button").click(timeout=5_000)
    except Exception:
        return False

    await page.wait_for_timeout(2_000)
    await page.evaluate("window.scrollTo(0, 0)")
    return True


async def scrapear_anuncios(url, max_scrolls=30, max_paginas=None):
    anuncios = {}

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page(
            viewport={"width": 1440, "height": 900},
            locale="es-CO",
        )

        print("Abriendo la página...")
        await page.goto(url, wait_until="domcontentloaded", timeout=60_000)
        await page.wait_for_timeout(5_000)

        try:
            await page.get_by_role("button", name="Aceptar").click(timeout=3_000)
        except Exception:
            pass

        numero_pagina = 1
        while True:
            if max_paginas is not None and numero_pagina > max_paginas:
                print(f"Se alcanzó el límite de {max_paginas} páginas.")
                break

            print(f"--- Página {numero_pagina} ---")
            await recolectar_anuncios_de_pagina(page, anuncios, max_scrolls)

            hay_siguiente = await ir_a_siguiente_pagina(page)
            if not hay_siguiente:
                print("No hay más páginas.")
                break

            numero_pagina += 1

        await browser.close()

    return pd.DataFrame(anuncios.values())

## 4. Ejecutar la extracción

En Jupyter se usa `await` porque Playwright está utilizando su API asíncrona. Se abrirá una ventana de Chromium.

In [5]:
df = ejecutar_coroutine_en_hilo(lambda: scrapear_anuncios(URL, MAX_SCROLLS, MAX_PAGINAS))
print(f"Total de anuncios: {len(df)}")
df.head()

C:\Users\USER\AppData\Local\Temp\ipykernel_35140\3289346000.py:33: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\USER\AppData\Local\Temp\ipykernel_35140\3289346000.py:33: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


Abriendo la página...
--- Página 1 ---
  Intento 1: 6 anuncios en total
  Intento 2: 14 anuncios en total
  Intento 3: 20 anuncios en total
  Intento 4: 26 anuncios en total
  Intento 5: 35 anuncios en total
  Intento 6: 44 anuncios en total
  Intento 7: 50 anuncios en total
  Intento 8: 59 anuncios en total
  Intento 9: 65 anuncios en total
  Intento 10: 68 anuncios en total
  Intento 11: 68 anuncios en total
  Intento 12: 68 anuncios en total
--- Página 2 ---
  Intento 1: 73 anuncios en total
  Intento 2: 81 anuncios en total
  Intento 3: 87 anuncios en total
  Intento 4: 93 anuncios en total
  Intento 5: 93 anuncios en total
  Intento 6: 102 anuncios en total
  Intento 7: 117 anuncios en total
  Intento 8: 117 anuncios en total
  Intento 9: 126 anuncios en total
  Intento 10: 135 anuncios en total
  Intento 11: 135 anuncios en total
  Intento 12: 135 anuncios en total
--- Página 3 ---
  Intento 1: 135 anuncios en total
  Intento 2: 135 anuncios en total
--- Página 4 ---
  Intento 1:

,url,texto,precio_texto,area_m2,habitaciones,banos,parqueaderos,tipo,sector,ciudad
0,https://www.metrocuadrado.com/inmueble/arriend...,Destacado Santa Barbara | Bogotá D.C. $25.000....,$25.000.000,492,NaN,5,4,NaN,Santa Barbara,Bogotá D.C. Foto de Oficina en Arriendo en San...
1,https://www.metrocuadrado.com/inmueble/arriend...,Destacado Chapinero Occidental | Bogotá D.C. $...,$2.000.000,66,NaN,1,2,NaN,Chapinero Occidental,Bogotá D.C. Foto de Oficina en Arriendo en Cha...
2,https://www.metrocuadrado.com/inmueble/arriend...,Destacado El Escobero | Envigado $7.000.000 Ap...,$7.000.000,93,2,5,NaN,Apartamento en Arriendo,El Escobero,Envigado Foto de Apartamento en Arriendo en El...
3,https://www.metrocuadrado.com/inmueble/arriend...,NORMANDIA Zona Urbana | Occidental | Bogotá D....,$7.000.000,130,NaN,2,2,NaN,NORMANDIA Zona Urbana,Bogotá D.C. Foto de Bodega en Arriendo en NORM...
4,https://www.metrocuadrado.com/inmueble/arriend...,CIUDAD SALITRE Zona Urbana | Occidental | Bogo...,$4.700.000,42,NaN,1,2,NaN,CIUDAD SALITRE Zona Urbana,Bogotá D.C. Foto de Oficina en Arriendo en CIU...


## 5. Guardar los resultados

In [6]:
df.to_csv("anuncios_cali.csv", index=False, encoding="utf-8-sig")
df.to_excel("anuncios_cali.xlsx", index=False)
print("Archivos creados: anuncios_arriendo.csv y anuncios_arriendo.xlsx")

Archivos creados: anuncios_arriendo.csv y anuncios_arriendo.xlsx


## 6. Revisar y filtrar

Ejemplo: mostrar anuncios cuyo texto contiene `PANCE`.

In [7]:
df[df["texto"].str.contains("PANCE", case=False, na=False)][
    ["precio_texto", "area_m2", "habitaciones", "tipo", "url"]
]

,precio_texto,area_m2,habitaciones,tipo,url
32,$5.000.000,117,3,Apartamento en Arriendo,https://www.metrocuadrado.com/inmueble/arriend...
88,$12.500.000,500,3,Casa en Arriendo,https://www.metrocuadrado.com/inmueble/arriend...
